# **Laboratórios - Aula 27/08/2026**

*Por: Ariel Reises*

# **Laboratório 01**

**Criando "níveis" de risco em vez de sintomas idênticos**

* **Grupo 1 (Sintomas 10 e 20):** Um grupo menor com uma taxa maior de doentes
* **Grupo 2 (Sintomas 1 e 2):** A grande massa de pessoas, quase todos saudáveis

In [ ]:
# Dataset Lab 01
dados = {
    'sintoma_x': [10]*10 + [1]*90,
    'sintoma_y': [20]*10 + [2]*90,
    'doente': [1, 1, 1, 0, 0, 0, 0, 0, 0, 0] + [1, 1] + [0]*88
}

In [ ]:
# Importações
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# Criação do Dataframe
df = pd.DataFrame(dados)

In [ ]:
# Separando as features (X) e target (y)
X = df.drop('doente', axis=1)
y = df['doente']

In [ ]:
# Split dos dados em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Criando o modelo
modelo = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

In [ ]:
# Treinando
modelo.fit(X_treino, y_treino)

In [ ]:
# Fazendo previsões
previsao = modelo.predict(X_teste)

In [ ]:
# Calculando acurácia
acuracia = accuracy_score(y_teste, previsao)
print(f"Acurácia do modelo: {acuracia*100:.2f}%")

## **Matriz de Confusão e Relatório de Classificação**

In [ ]:
# Importando a Matriz de confusão e Relatório de Classificação do Scikit-Learn
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Gerando a matriz de confusão
matriz_confusao = confusion_matrix(y_teste, previsao, labels=[0, 1])

In [ ]:
# Configurando o plot
plt.figure(figsize=(8, 6))
plt.title('Matriz de Confusão')
sns.heatmap(matriz_confusao, annot=True, fmt='g', cmap='Reds',
            xticklabels=['Previsto: Saudável(0)', 'Previsto: Doente (1)'],
            yticklabels=['Realidade: Doente(0)', 'Previsto: Doente(1)'])
plt.xlabel('Previsto')
plt.ylabel('Realidade')

# Plotando a Matriz
plt.show()

In [ ]:
# Relatório de Classificação
relatorio_classificacao = classification_report(y_teste, previsao, zero_division=0, target_names=['Saudável (0)', 'Doente (1)'])
print(relatorio_classificacao)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Obtendo as probabilidades
probabilidades = modelo.predict_proba(X_teste)[:,1]

# Calculando a Curva ROC
fpr, tpr, cortes = roc_curve(y_teste, probabilidades)

# Calculando AUC
roc_auc = auc(fpr, tpr)

print(f"AUC: {roc_auc:.2f}")

In [ ]:
# Plotando
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkblue', marker='o', lw=2, label=f'Curva ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='orange', lw=2, linestyle='--')

for i in range(len(cortes)):
    texto = "Corte: Padrão" if cortes[i] > 1 else f"Corte: {cortes[i]:.2f}"
    plt.annotate(texto, (fpr[i], tpr[i]),
                 textcoords="offset points",
                 xytext=(15, -15),
                 ha='left',
                 fontsize=10,
                 color='darkred',
                 weight='bold')

plt.title('Mapa de Decisão')
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos')
plt.xlim([-0.05, 1.05])
plt.ylim([-0.05, 1.05])
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Alterando o ponto de corte
ponto_de_corte = 0.12
novas_previsoes = [1
                   if risco >= ponto_de_corte
                   else 0 for risco in probabilidades]

In [ ]:
# Nova Matriz de Confusão
nova_matriz = confusion_matrix(y_teste, novas_previsoes, labels=[0, 1])

In [ ]:
# Plotando a nova matriz
plt.figure(figsize=(8, 6))
plt.title('Nova Matriz de Confusão')
sns.heatmap(nova_matriz, annot=True, fmt='g', cmap='Greens',
            xticklabels=['Previsto: Saudável(0)', 'Previsto: Doente (1)'],
            yticklabels=['Realidade: Doente(0)', 'Previsto: Doente(1)'])
plt.xlabel('Previsto')
plt.ylabel('Realidade')

# Plotando a Matriz
plt.show()

In [ ]:
# Novo Relatório de Classificação
relatorio_classificacao = classification_report(y_teste, novas_previsoes, zero_division=0, target_names=['Saudável (0)', 'Doente (1)'])
print(relatorio_classificacao)

#
---

# **Laboratório 02**

**Uma base de dados rápida para a máquina ter o que fatiar** *(10 pacientes)*

*Assim fica óbvio ver o Cross-Validation cortando blocos de 2 em 2.*

In [ ]:
# Dataset Lab 02
df_cv = pd.DataFrame({
'pressao': [120, 180, 130, 160, 110, 190, 140, 175, 125, 165],
'febre':   [36,   39,   36.5, 38,   36,   40,   37,   38.5, 36.5, 39],
'doente':  [0,    0,    0,    1,    0,    1,    0,    1,    0,    1]
})

In [ ]:
# Importações
import pandas as pd

In [ ]:
# Importações do Scikit-Learn
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

In [ ]:
# Separação das features
X_cv = df_cv.drop('doente', axis=1)
y_cv = df_cv['doente']

In [ ]:
# Criando o modelo
modelo_cv = DecisionTreeClassifier(max_depth=2,
                                   random_state=42)

In [ ]:
# Fazendo o Cross Validation
notas = cross_val_score(modelo_cv,
                        X_cv,
                        y_cv,
                        cv=4, # Diminuí pra 4 pq tava dando warn... Antes era 5
                        )

In [ ]:
# Exibindo os resultados
print(f"Acurácia de cada execução de folds: {notas * 100}")

In [ ]:
# Calculando a média
media = notas.mean()
print(f"Acurácia média: {media*100:.2f}%")

#
---

# **Laboratório 03**

**Os dados brutos** *(Uma simulação rápida de um modelo que já operou)*


**`y_real`**: `0` é **Transação Segura**, `1` é **Fraude** *(Gabarito)*

In [ ]:
y_real = [0, 1, 0, 1, 0, 1, 1, 0]

`risco_calculado`: A IA não diz *Sim* ou *Não*, **ela dá a %** de certeza do golpe

In [ ]:
risco_calculado = [0.1, 0.8, 0.4, 0.9, 0.2, 0.6, 0.3, 0.7]

print("A Mente da IA:")
for i in range(len(y_real)):
    print(f"Transação {i+1} -> Realidade: {' Fraude ' if y_real[i]==1 else '   Ok   '} | Certeza da IA: {risco_calculado[i]*100:.2f}%")